# NEEZY on Kaggle

Install and run **[NEEZY](https://github.com/jvrboy/NEEZY)** — a local-first Python
market-monitoring workspace — inside a Kaggle notebook. NEEZY reads public candles from
Deriv's WebSocket feed and turns them into educational technical-analysis summaries
(confluence cards, strategy scores, backtests) that it stores in a local SQLite file.

> NEEZY places no trades, never logs into a broker account, and gives no financial
> advice. Everything here analyses public market data for learning and research.

### What works inside Kaggle

| NEEZY component | On Kaggle | Why |
| --- | --- | --- |
| Headless engine + `neezy-cli` (analysis, confluence cards, scanner, backtests) | ✅ | pure Python |
| Local SQLite store (`NEEZY_HOME`) | ✅ | point it at `/kaggle/working` to survive the session |
| Deriv public market data | ✅ with **Internet ON** | outbound WebSocket on 443 |
| BYO-key AI providers (Gemini / Grok / OpenRouter / Ollama Cloud) | ✅ | keys from Kaggle Secrets |
| Optional local GGUF / Transformers models | ⚠️ | CPU only, extra native wheels, slow |
| Desktop Tk workspace | ⚠️ screenshot only | Kaggle has no display; the GUI cell renders a PNG via Xvfb |
| MT5 bridge, Telegram bot, local MCP server, Cloudflare tunnel | ❌ | needs Windows MetaTrader, a persistent bot host, or a loopback client |

### Before you run

1. Open **Settings → Internet** in the notebook editor and switch it **ON**
   (new Kaggle accounts must verify a phone number first).
2. **Run all**. Installation takes about 20 seconds; the optional desktop screenshot
   adds another 30–60 seconds.
3. No GPU is needed — leave the accelerator off.

No Internet? Attach this repository as a Kaggle **Dataset** and set
`INSTALL_SOURCE = "dataset"` in the next cell. `kaggle/README.md` in the repository
has the exact steps.

## 1 · Settings

Everything the notebook does is driven by this cell. `NEEZY_HOME` is set **before**
anything imports NEEZY so that every CLI subprocess writes to the same place.

In [ ]:
# =============================================================================
# Settings - edit these, then Run All
# =============================================================================
import os
import shlex
import subprocess
import sys
from pathlib import Path

GITHUB_REPO    = "jvrboy/NEEZY"   # a public fork works too
GIT_REF        = "main"           # branch, tag or commit; "" = repository default branch
INSTALL_SOURCE = "auto"           # "auto" | "github" | "dataset"

NEEZY_HOME     = "/kaggle/working/neezy-home"   # /kaggle/working survives the session
WATCHLIST      = "frxEURUSD,frxGBPUSD,frxUSDJPY,cryBTCUSD"
TIMEFRAME      = "15m"
CANDLE_COUNT   = 300

RUN_TESTS      = False            # ~30 s: runs the repository's 367-test suite
SHOW_GUI_SHOT  = True             # ~40 s: renders the Tk desktop app under Xvfb
# =============================================================================

if not Path("/kaggle").exists():                 # running outside Kaggle?
    NEEZY_HOME = str(Path.home() / ".neezy")
HOME = Path(NEEZY_HOME).expanduser()
HOME.mkdir(parents=True, exist_ok=True)
os.environ["NEEZY_HOME"] = str(HOME)             # must be set before NEEZY is imported


def clip(text, width=280):
    """Keep long single-line CLI errors readable in the notebook."""
    return "\n".join(
        line if len(line) <= width else line[:width] + " ..."
        for line in (text or "").strip().splitlines()
    )


def neezy(*args, timeout=600, quiet=False, expect=(0,)):
    """Run the NEEZY CLI with this interpreter (never relies on PATH)."""
    command = [sys.executable, "-m", "neezy_backend", *args]
    if not quiet:
        print("$ neezy-cli " + " ".join(shlex.quote(str(arg)) for arg in args))
    try:
        proc = subprocess.run(command, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"  ! timed out after {timeout}s")
        return None
    if proc.stdout.strip() and not quiet:
        print(clip(proc.stdout))
    if proc.returncode not in expect:
        # stdout may be a JSON payload the caller parses, so only stderr is shown
        print(clip(proc.stderr) or f"  ! exit code {proc.returncode}")
    return proc


print("NEEZY_HOME :", HOME)
print("Python     :", sys.version.split()[0])
print("Source     :", INSTALL_SOURCE, f"({GITHUB_REPO} @ {GIT_REF or 'default branch'})")

## 2 · Install

Kaggle gives every session a fresh container and installed packages do **not** survive a
restart, so this cell has to run each time (Run All does that for you).

* **GitHub mode** — `pip install git+https://github.com/<repo>.git` (needs Internet).
* **Dataset mode** — if a copy of the repository is attached as a Kaggle Dataset, NEEZY is
  pure Python, so the checkout is added to `sys.path`. No build backend, no downloads, and
  it keeps working with Internet switched off. This mode does not create the `neezy-cli`
  console script; use `python -m neezy_backend ...` (the `neezy()` helper above already
  does that) or the Python API shown further down.

In [ ]:
# Install NEEZY ---------------------------------------------------------------
import shutil


def run(command, **kwargs):
    print("$", " ".join(str(part) for part in command))
    proc = subprocess.run(command, capture_output=True, text=True, **kwargs)
    tail = (proc.stdout + proc.stderr).strip()
    if tail:
        print("\n".join(tail.splitlines()[-8:]))
    return proc


def find_attached_repo():
    """Locate a NEEZY checkout uploaded as a Kaggle Dataset."""
    candidates = []
    for base in (Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()):
        if not base.is_dir():
            continue
        candidates.append(base / "pyproject.toml")
        candidates.extend(sorted(base.glob("*/pyproject.toml")))
        candidates.extend(sorted(base.glob("*/*/pyproject.toml")))
    for pyproject in candidates:
        root = pyproject.parent
        if (root / "neezy_backend").is_dir() and (root / "neezy_app").is_dir():
            return root
    return None


attached = find_attached_repo() if INSTALL_SOURCE in ("auto", "dataset") else None
installed = False

if attached is not None:
    print(f"Using the attached checkout: {attached}")
    sys.path.insert(0, str(attached))
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(attached), os.environ.get("PYTHONPATH", "")]
    ).strip(os.pathsep)
    try:
        import websocket  # noqa: F401  (websocket-client, NEEZY's only runtime dependency)

        print("websocket-client: present")
    except ModuleNotFoundError:
        print("websocket-client is missing - trying to install it (needs Internet)")
        run([sys.executable, "-m", "pip", "install", "--quiet", "websocket-client"])
    installed = True
elif INSTALL_SOURCE == "dataset":
    print("INSTALL_SOURCE='dataset' but no checkout was found under /kaggle/input.")
    print("Attach the repository as a Dataset, or set INSTALL_SOURCE = 'auto'.")

if not installed and INSTALL_SOURCE in ("auto", "github"):
    ref = f"@{GIT_REF}" if GIT_REF else ""
    spec = f"git+https://github.com/{GITHUB_REPO}.git{ref}"
    proc = run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-input",
            "--disable-pip-version-check",
            spec,
        ]
    )
    installed = proc.returncode == 0

if not installed:
    raise RuntimeError(
        "NEEZY is not installed.\n"
        "  * Kaggle: Notebook options -> Internet -> On (a phone-verified account is required).\n"
        "  * No Internet: attach the repository as a Dataset and set INSTALL_SOURCE = 'dataset'.\n"
        "  * Private fork: publish it, or pin a tag/commit in GIT_REF.\n"
        "Then run this cell again."
    )

print("NEEZY is ready.")

## 3 · Check the installation and the market-data connection

`neezy-cli markets` is fully offline; `neezy-cli doctor` probes Deriv's public endpoints.
Every endpoint failing means the network path is blocked, not that the app is broken —
NEEZY rotates endpoints automatically and `NEEZY_WS_URL` can pin one that works.

In [ ]:
import json

try:
    import neezy_backend

    print("package    :", Path(neezy_backend.__file__).parent)
    print("engine     :", neezy_backend.__version__)
except ModuleNotFoundError as exc:
    raise SystemExit(f"NEEZY is not importable ({exc}); re-run the install cell.") from exc

version = neezy("--version", quiet=True)
print("CLI        :", (version.stdout or version.stderr).strip())

markets = neezy("markets", "--json", quiet=True)
try:
    catalogue = json.loads(markets.stdout)
except json.JSONDecodeError:
    print(markets.stderr.strip() or "neezy-cli markets failed")
    catalogue = {"markets": {}, "categories": [], "timeframes": {}}
table = catalogue.get("markets", {})
print(f"markets    : {len(table)} across {', '.join(catalogue.get('categories', []))}")
print("timeframes :", ", ".join(catalogue.get("timeframes", {})))
for symbol in list(table)[:5]:
    print(f"             {symbol:12s} {table[symbol]}")

print("\nConnectivity check (Deriv public market data):")
# exit code 1 simply means "no endpoint reachable"; the JSON report is still valid
doctor = neezy("doctor", "--json", quiet=True, expect=(0, 1))
try:
    endpoints = json.loads(doctor.stdout or "[]")
except json.JSONDecodeError:
    endpoints = []
for endpoint in endpoints:
    mark = "reachable" if endpoint.get("reachable") else "unreachable"
    print(f"  {mark:11s} {endpoint.get('url')}")
if not any(endpoint.get("reachable") for endpoint in endpoints):
    print(
        "  No endpoint answered. Check that Internet is ON for this notebook, then\n"
        "  try a different network (mobile hotspot, no VPN/proxy). NEEZY rotates the\n"
        "  endpoints itself; 'NEEZY_WS_URL' can pin a working one."
    )

## 4 · Live confluence cards

This is what the desktop app's **SIGNALS → ANALYSIS** button runs: every backend tool
(EMA stack, MACD, RSI, ADX/DI, Bollinger extremes, all named strategies, divergence,
learned weights) votes, and the weighted net becomes one card per market —
**INSTRUMENT · BUY/SELL · ENTRY · SL · TP · WIN RATE**.

The win rate is an in-sample replay of the market's own history, not a forecast. The
`MODE` column shows which pass produced the card (`strict` = score above the strict
threshold, `search` = the relaxed pass that keeps weak setups cardable).

In [ ]:
from neezy_backend.markets import DEFAULT_WATCHLIST, SYMBOLS, display_name
from neezy_backend.providers import get_candles
from neezy_backend.scanner import ConfluenceAnalyzer
from neezy_backend.storage import SettingsStore

settings = SettingsStore().load()
provider = settings.market_data_provider


def canonical(value):
    """Accept EURUSD, EUR/USD or frxEURUSD and return the canonical symbol."""
    needle = value.replace("/", "").strip().upper()
    for symbol in SYMBOLS:
        if needle in (symbol.upper(), display_name(symbol).replace("/", "").upper()):
            return symbol
    return needle.lower()


symbols = [canonical(item) for item in WATCHLIST.split(",") if item.strip()]
symbols = symbols or list(DEFAULT_WATCHLIST[:5])
print(f"provider {provider} · {TIMEFRAME} · {len(symbols)} markets · {CANDLE_COUNT} candles each")


def candle_source(symbol, timeframe):
    return get_candles(provider, symbol, timeframe, count=CANDLE_COUNT)


result = ConfluenceAnalyzer(candle_source, settings).full_analysis(symbols, TIMEFRAME)
print(result.summary())

rows = [
    {
        "INSTRUMENT": signal.instrument,
        "SIDE": signal.label,
        "ENTRY": round(signal.entry, 5),
        "SL": round(signal.stop_loss, 5),
        "TP": round(signal.take_profit, 5),
        "WIN RATE": f"{signal.win_rate:.1f}%",
        "SCORE": round(signal.score, 1),
        "MODE": signal.mode,
        "WIN RATE BASIS": signal.win_rate_basis,
    }
    for signal in result.signals
]

if rows:
    try:
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame(rows).set_index("INSTRUMENT"))
    except Exception:
        for row in rows:
            print(row)
else:
    print("No cards produced (no usable candle data reached the engine).")

if result.errors:
    print("\nMarkets that could not be analysed (network or provider problem):")
for symbol, error in result.errors.items():
    print(f"  ! {clip(error, 240)}")  # the message already starts with the instrument name

## 5 · Store candles, snapshots and history locally

NEEZY keeps everything in one SQLite file under `NEEZY_HOME` — here
`/kaggle/working/neezy-home`, which Kaggle persists as notebook output when you
**Save Version → Save & Run All**.

In [ ]:
neezy("db", "save", "EURUSD", "-t", TIMEFRAME, "-n", "600")
neezy("db", "stats")
neezy("history", "--summary")

print("\nFiles in NEEZY_HOME:")
for path in sorted(HOME.iterdir()):
    size = path.stat().st_size if path.is_file() else 0
    print(f"  {path.name:26s} {size:>12,} bytes")

## 6 · Backtest and the position-size calculator

The backtester decides on closed candles only, fills at the next candle's open, and
assumes the stop fills first when both stop and target sit inside one candle. `risk`
works offline, so it is a good check that the engine is installed correctly.

In [ ]:
neezy("backtest", "EURUSD", "-t", "5m", "-n", "2000", "--spread-pips", "0.8", timeout=900)

neezy(
    "risk",
    "EURUSD",
    "--balance",
    "10000",
    "--risk",
    "1",
    "--entry",
    "1.0850",
    "--stop",
    "1.0825",
)

## 7 · Optional: the real desktop workspace as a screenshot

Kaggle notebooks have no display, so the Tk app cannot be *used* here — but it can be
rendered on a virtual X server and captured. This cell installs `xvfb` + ImageMagick
through `apt` (the notebook runs as root), launches `python -m neezy_app`, waits for its
window, and saves `/kaggle/working/neezy-workspace.png`.

It degrades quietly: if this image has no `tkinter`, no root access, or no `apt`, the rest
of the notebook is unaffected.

In [ ]:
import time


def has_tkinter():
    return (
        subprocess.run(
            [sys.executable, "-c", "import tkinter"], capture_output=True
        ).returncode
        == 0
    )


def apt_install(packages):
    is_root = not hasattr(os, "geteuid") or os.geteuid() == 0
    prefix = [] if is_root else ["sudo", "-n"]
    if prefix and subprocess.run(prefix + ["true"], capture_output=True).returncode != 0:
        print("No root or passwordless sudo - cannot install a virtual display.")
        return False
    subprocess.run(prefix + ["apt-get", "update", "-qq"], capture_output=True, text=True)
    proc = subprocess.run(
        prefix + ["apt-get", "install", "-y", "-qq", *packages],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout).strip().splitlines()[-3:]
        print("apt-get failed:", " ".join(tail))
    return proc.returncode == 0


if not SHOW_GUI_SHOT:
    print("SHOW_GUI_SHOT = False - skipped.")
elif not has_tkinter():
    print("This Python has no tkinter module, so the Tk desktop app cannot start here.")
    print("The headless engine and CLI above do not need it; only this screenshot does.")
else:
    shot_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else HOME
    shot = shot_dir / "neezy-workspace.png"
    log_path = shot_dir / "neezy-gui.log"

    tools = ("Xvfb", "xwininfo", "import")
    if any(shutil.which(tool) is None for tool in tools):
        print("Installing a virtual display and a screenshot tool ...")
        apt_install(["xvfb", "x11-utils", "imagemagick", "fonts-dejavu-core"])

    if any(shutil.which(tool) is None for tool in tools):
        print("Xvfb / xwininfo / ImageMagick is unavailable - skipping the screenshot.")
    else:
        gui_env = {**os.environ, "DISPLAY": ":99"}
        with log_path.open("w") as log:
            xvfb = subprocess.Popen(
                ["Xvfb", ":99", "-screen", "0", "1440x900x24", "-nolisten", "tcp"],
                stdout=log,
                stderr=log,
            )
            app = None
            try:
                time.sleep(2)
                app = subprocess.Popen(
                    [sys.executable, "-m", "neezy_app"],
                    env=gui_env,
                    stdout=log,
                    stderr=log,
                )
                deadline = time.time() + 60
                while time.time() < deadline:
                    if app.poll() is not None:
                        break
                    tree = subprocess.run(
                        ["xwininfo", "-root", "-children"],
                        env=gui_env,
                        capture_output=True,
                        text=True,
                    ).stdout
                    if "NEEZY" in tree:
                        break
                    time.sleep(2)
                time.sleep(5)  # first chart frame + market-data handshake
                subprocess.run(
                    ["import", "-window", "root", str(shot)],
                    env=gui_env,
                    capture_output=True,
                )
            finally:
                for proc in (app, xvfb):
                    if proc is not None and proc.poll() is None:
                        proc.terminate()
                        try:
                            proc.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            proc.kill()

        if shot.exists():
            print(f"Screenshot saved to {shot}")
            try:
                from IPython.display import Image, display

                display(Image(filename=str(shot)))
            except Exception:
                pass
        else:
            print("No screenshot produced. Log tail:")
            print("\n".join(log_path.read_text(errors="replace").splitlines()[-15:]))

## 8 · Optional: run the repository's test suite

The package installed from GitHub does not ship `tests/`, so this cell uses an attached
checkout when there is one, otherwise it makes a shallow clone. It is the same suite CI
runs on Python 3.10–3.12.

In [ ]:
if not RUN_TESTS:
    print("RUN_TESTS = False - skipped (set it to True in the settings cell).")
else:
    scratch = next(
        (path for path in (Path("/kaggle/temp"), Path("/kaggle/working")) if path.is_dir()),
        Path.home() / ".neezy-src",
    )
    source = find_attached_repo() or (scratch / "NEEZY")
    if not (source / "tests").is_dir():
        if shutil.which("git") is None:
            print("git is unavailable and no checkout is attached - skipping.")
            source = None
        else:
            print(f"Cloning {GITHUB_REPO} for its test suite ...")
            run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    f"https://github.com/{GITHUB_REPO}.git",
                    str(source),
                ]
            )
    if source is not None and (source / "tests").is_dir():
        env = {key: value for key, value in os.environ.items() if key != "PYTHONPATH"}
        proc = subprocess.run(
            [sys.executable, "-m", "unittest", "discover", "-s", "tests"],
            cwd=source,
            env=env,
            capture_output=True,
            text=True,
            timeout=1800,
        )
        print("\n".join((proc.stdout + proc.stderr).strip().splitlines()[-12:]))
        print("exit code:", proc.returncode)

## 9 · Optional: AI provider keys from Kaggle Secrets

NEEZY's chat is bring-your-own-key: Gemini, Grok (xAI), OpenRouter and Ollama Cloud.
Store the key as a Kaggle secret instead of pasting it into a cell —
**Add-ons → Secrets → Add secret**, label `NEEZY_AI_KEY` — then run this cell.

Market-data provider keys (`ALPHA_VANTAGE_KEY`, `FINNHUB_KEY`, `TWELVE_DATA_KEY`) work the
same way; Deriv's public feed needs no key at all.

In [ ]:
SECRET_NAMES = ("NEEZY_AI_KEY", "ALPHA_VANTAGE_KEY", "FINNHUB_KEY", "TWELVE_DATA_KEY")
loaded = {}

try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
except Exception as exc:  # noqa: BLE001 - only available inside Kaggle
    client = None
    print(f"Kaggle Secrets not available here ({exc.__class__.__name__}).")

if client is not None:
    for name in SECRET_NAMES:
        try:
            value = client.get_secret(name)
        except Exception:  # noqa: BLE001 - a missing secret raises
            continue
        if value:
            os.environ[name] = value
            loaded[name] = len(value)
    print("loaded:", loaded or "nothing (no matching secrets found)")

if loaded.get("NEEZY_AI_KEY"):
    print(
        "\nRegister the key with one provider, for example:\n"
        "  !python -m neezy_backend ai keys add gemini --label kaggle-secret\n"
        "then:\n"
        '  !python -m neezy_backend ai chat "is EURUSD trending?" --provider gemini --symbol EURUSD'
    )
    neezy("ai", "providers")
else:
    print("\nNo AI key loaded - the rest of NEEZY works without one.")

## 10 · Your data, and keeping it

| Location | Contents |
| --- | --- |
| `/kaggle/working/neezy-home/neezy.sqlite3` | candles, analysis snapshots, signal history, AI call log, local notes |
| `/kaggle/working/neezy-home/settings.json` | workspace / engine / data-provider preferences |
| `/kaggle/working/neezy-home/provider-keys.json` | market-data API keys you save from the app |
| `/kaggle/working/neezy-home/ai-keys.json` | AI provider keys added with `ai keys add` |

* **Installed packages** do not persist between sessions — re-run the install cell (or
  Run All) after every restart.
* **`/kaggle/working`** does persist when you run **Save Version → Save & Run All**, up to
  20 GB, and appears in the notebook's *Output* tab. You can re-attach an old output as an
  input to keep your database.
* **Sessions are limited** (12 h CPU/GPU, 9 h TPU) and stop when you close the tab. Save
  snapshots with `neezy-cli db export /kaggle/working/neezy-export.json` if you want a
  portable copy.
* **Secret hygiene:** if you register AI keys with `ai keys add`, they are written in plain
  text to `ai-keys.json` — and a saved notebook version would capture that file in its
  output. Prefer Kaggle Secrets and keep the notebook version private.
* **Retention:** `neezy-cli db prune --days 30` trims the local store.

## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `NEEZY is not installed` | Internet is off, the repo is private, or `GIT_REF` does not exist. Turn Internet on or attach the repo as a Dataset. |
| `neezy-cli markets` works but `doctor` shows every endpoint unreachable | Kaggle's egress is fine for Deriv in most regions; retry, disable a VPN on your side, then pin an endpoint with `os.environ["NEEZY_WS_URL"] = "wss://api.derivws.com/trading/v1/options/ws/public"` and re-run. |
| `ModuleNotFoundError: neezy_backend` after a restart | The container was reset. Re-run the install cell. |
| No screenshot produced | The Kaggle image lacked `tkinter`, or `apt-get` had no network/root. The CLI does not need it. |
| `No cards produced` with `! market unavailable` lines | The engine got no candles: connectivity or an unsupported market/timeframe. |
| Output stops mid-run | Kaggle killed the session at its time limit — re-run and lean on the saved database. |

### What this notebook is *not*

NEEZY does not predict markets, does not place trades, and its win-rate figures are
in-sample replays of past candles. Treat every card as a description of the current
technical setup, not a recommendation. Trading involves substantial risk of loss.